# 繰り返し平均によるランキング

同じ入力で 10 回まわしても、このパイプラインは 10 回とも同じ答えを返します。温度 0 で復号しているからです。
測る価値のあるばらつきは別の場所にあります。

段階 2 は候補を **4 個ずつのグループに切って** 出題し、各遺伝子は同じグループの 3 個としか比較されません。
どの 3 個と同組になるかを決めているのは入力の並び順だけなので、無名の遺伝子 3 個と当たれば高득点、
DRD2 と当たれば低得点になります。ローテーションは選択肢の *位置* を制御しますが、
グループの *構成* には何もしません。

そこでこのノートブックは、**毎回並び順をシャッフルして** 繰り返し、平均スコアで並べます。
併せて出すばらつき（SD・順位変動幅）は飾りではなく、**1 回の実行の順位をどこまで信じてよいか** の目安です。


## 0. 準備


In [ ]:
import os, sys, json, csv, importlib.util, statistics

SKILL = os.path.abspath(os.path.join("..", ".claude", "skills", "gene-disease-ranking"))
if not os.path.isdir(SKILL):
    SKILL = os.path.abspath(os.environ.get("GDR_SKILL_DIR", "."))
sys.path.insert(0, os.path.join(SKILL, "scripts"))
EXAMPLES = os.path.join(SKILL, "examples")

HAVE = {m: importlib.util.find_spec(m) is not None for m in ("numpy", "pandas")}
print("skill:", SKILL)
print("  " + "  ".join(f"{m}={'y' if ok else 'n'}" for m, ok in HAVE.items()))
if not HAVE["numpy"]:
    print("\nnumpy が必要です:  pip install numpy")


## 1. 入力 — ここだけ書き換える


In [ ]:
# ===== 疾患名 =====
DISEASE = "schizophrenia"

# ===== 候補遺伝子 =====
#   ファイルから読む場合はパスを、直接書く場合は GENES に並べる。
GENES_FILE = os.path.join(EXAMPLES, "schizo_candidates100.txt")   # None なら GENES を使う
GENES = []

# ===== どこで動かすか =====
BACKEND = "ollama"
MODEL   = None          # 例: "qwen3:14b" / "gemma3:27b"
OLLAMA_HOST = None      # None なら自動探索

# ===== 繰り返しの設定 =====
REPEATS    = 10   # 並び順を変えて何回まわすか。1 回あたり (候補数/4)*4 回の推論
GROUP_SIZE = 4    # 1 問あたりの遺伝子数（+ 逃げ道で A〜E）
ROTATIONS  = 4    # 1 グループ内で選択肢の順序を何通り試すか
SEED       = 0    # シャッフルの種。変えれば別の 10 通りになる
HGNC_PATH  = None # 例: "/path/to/hgnc_complete_set.txt"（別名の統一）

# ===== 外部ベースライン（任意）=====
#   {"遺伝子": スコア} の JSON。順位の答え合わせに使う。無ければ None。
BASELINE_JSON = os.path.join(EXAMPLES, "schizo_ot_association.json")
BASELINE_NAME = "Open Targets 関連スコア"

if GENES_FILE:
    with open(GENES_FILE) as f:
        GENES = [l.strip() for l in f if l.strip() and not l.startswith("#")]

n_calls = REPEATS * -(-len(GENES) // GROUP_SIZE) * ROTATIONS
print(f"疾患         : {DISEASE}")
print(f"候補         : {len(GENES)} 個")
print(f"繰り返し     : {REPEATS} 回  → 推論およそ {n_calls} 回")
print(f"モデル       : {MODEL or '★未設定 — ここを埋めないと動きません'}")


## 2. 接続の確認


In [ ]:
ranker = None

if not MODEL:
    print("MODEL が未設定です。1 章の MODEL に名前を入れてください。")
    print("  Ollama なら `ollama list` に出てくる名前（例: qwen3:14b）")
else:
    if BACKEND == "ollama" and OLLAMA_HOST is None:
        from backends import discover_ollama_host
        OLLAMA_HOST = discover_ollama_host()
        print(f"Ollama: {OLLAMA_HOST}" if OLLAMA_HOST else "Ollama が見つかりません。")

    from rank import GeneRanker
    kw = {"host": OLLAMA_HOST} if BACKEND == "ollama" and OLLAMA_HOST else {}
    if HGNC_PATH:
        kw["hgnc"] = HGNC_PATH
    ranker = GeneRanker(MODEL, backend=BACKEND, **kw)
    print(f"backend={BACKEND}  段階 1={'使える' if ranker.supports_pmi else '使えない'}")


## 3. まず「繰り返しても同じ」ことを確かめる

同じ並び順で 2 回まわします。差が 0 なら決定的です。
**ここで差が出ないことが、並び順をシャッフルする理由そのもの**なので、飛ばさないでください。
（差が出た場合は温度やサンプリングの設定が効いています。その場合は素直な繰り返しにも意味が生まれます。）


In [ ]:
from rank_repeats import repeat_rank, aggregate, instability

det = None
if ranker and len(GENES) >= 2:
    probe = GENES[:20]          # 確認用に先頭 20 個だけで十分
    a = ranker.rank(DISEASE, probe, group_size=GROUP_SIZE, rotations=ROTATIONS)
    b = ranker.rank(DISEASE, probe, group_size=GROUP_SIZE, rotations=ROTATIONS)
    sa = {r["gene"]: r["score"] for r in a["ranked"]}
    sb = {r["gene"]: r["score"] for r in b["ranked"]}
    det = max(abs(sa[g] - sb[g]) for g in sa)
    print(f"同じ並び順で 2 回: 最大スコア差 = {det:.3e}")
    print("→ 決定的。単純な繰り返しでは何も分からないので、並び順を変えます。"
          if det < 1e-12 else
          "→ 決定的ではありません。素直な繰り返しにも意味があります。")
else:
    print("モデル未設定のため確認できません。")


## 4. 並び順を変えて繰り返す

繰り返し 0 は元の並び順のままです。つまり `rank.py` を素で 1 回まわしたのと同じ結果になり、
後で「1 回だけの実行」と比較するときの基準になります。


In [ ]:
per_run = []

def progress(run):
    call = run["call"] or "なし（棄権）"
    print(f'  繰り返し {run["repeat"]:>2}: 1 位={run["top"]:<10} '
          f'差={run["margin"]}  判定={call}', flush=True)

if ranker:
    per_run = repeat_rank(ranker, DISEASE, GENES, repeats=REPEATS,
                          group_size=GROUP_SIZE, rotations=ROTATIONS,
                          seed=SEED, progress=progress)
else:
    print("モデル未設定のため実行しません。")


## 5. 平均スコアのランキング


In [ ]:
def show(rows, cols=None, limit=None):
    rows = rows[:limit] if limit else rows
    if not rows:
        print("（該当なし）"); return
    cols = cols or list(rows[0])
    if HAVE["pandas"]:
        import pandas as pd
        df = pd.DataFrame(rows, columns=cols)
        try:
            display(df)
        except NameError:
            print(df.to_string(index=False))
        return
    w = [max(len(str(c)), *(len(str(r.get(c, ""))) for r in rows)) for c in cols]
    head = "  ".join(str(c).ljust(x) for c, x in zip(cols, w))
    print(head); print("-" * len(head))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(x) for c, x in zip(cols, w)))


rows = aggregate(per_run, GENES) if per_run else []

table = [{
    "順位": r["rank"], "遺伝子": r["gene"],
    "平均スコア": round(r["mean_score"], 3),
    "SD": round(r["sd_score"], 3),
    "順位変動": f"{r['best_rank']}-{r['worst_rank']}",
    "順位SD": round(r["rank_sd"], 1),
} for r in rows]

show(table, limit=30)


点数の読み方：

- **平均スコア** は繰り返しをまたいだ平均対数確率です。絶対値に意味はなく、比較だけに使います。
- **SD / 順位変動 / 順位SD** が大きい遺伝子は、グループ構成しだいで順位が動くということです。
  変動幅が 50 位を超えるような遺伝子は、1 回の実行の順位を根拠にしてはいけません。


## 6. 1 回だけの実行はどれくらい当てにならないか


In [ ]:
if rows:
    inst = instability(rows)
    print(f"順位SD の中央値          : {inst['median_rank_sd']:.1f} 位")
    print(f"順位変動幅の中央値       : {inst['median_rank_span']:.0f} 位")
    print(f"変動幅が 50 位超の遺伝子 : {inst['n_span_over_50']}/{inst['n_genes']}")

    first = per_run[0]
    order1 = sorted(first["scores"], key=lambda g: -first["scores"][g])
    r1 = {g: i + 1 for i, g in enumerate(order1)}
    moved = sorted(rows, key=lambda r: -abs(r1[r["gene"]] - r["rank"]))[:10]
    print("\n1 回目と平均で最も動いた 10 個:")
    show([{
        "遺伝子": r["gene"], "1 回目": r1[r["gene"]], "平均": r["rank"],
        "差": r["rank"] - r1[r["gene"]],
    } for r in moved])

    calls = [r for r in per_run if r["call"]]
    print(f"\n判定を返した回: {len(calls)}/{len(per_run)}")
    from collections import Counter
    print("1 位になった遺伝子:", dict(Counter(r["top"] for r in per_run)))


## 7. 外部ベースラインとの答え合わせ

モデルの順位だけを眺めても良し悪しは分かりません。独立した根拠と突き合わせます。
同梱の既定値は Open Targets の疾患-遺伝子関連スコアです。

相関が高くても喜ぶ前に、**文献量との相関** を見てください。よく研究された遺伝子ほど
関連スコアも論文数も高くなりがちで、モデルが後者を写しているだけということがあります。


In [ ]:
def spearman(x, y):
    def rk(a):
        idx = sorted(range(len(a)), key=lambda i: a[i])
        r = [0] * len(a)
        for p, i in enumerate(idx):
            r[i] = p
        return r
    rx, ry = rk(x), rk(y)
    n = len(x)
    mx, my = sum(rx) / n, sum(ry) / n
    num = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    den = (sum((a - mx) ** 2 for a in rx) * sum((b - my) ** 2 for b in ry)) ** 0.5
    return num / den if den else 0.0


base = {}
if BASELINE_JSON and os.path.exists(BASELINE_JSON):
    base = json.load(open(BASELINE_JSON))

if rows and base:
    g = [r["gene"] for r in rows]
    m = [r["mean_score"] for r in rows]
    b = [base.get(x, 0) for x in g]
    print(f"Spearman(平均スコア, {BASELINE_NAME}) = {spearman(m, b):+.3f}")

    pos = {x: i + 1 for i, x in enumerate(g)}
    top = [x for x, _ in sorted(base.items(), key=lambda kv: -kv[1]) if x in pos]
    n = len(g)
    # 打ち切りが候補数以上だと比較にならないので、候補数より小さいものだけ見る
    ks = [k for k in (5, 10, 20) if k < n]
    ms = [m_ for m_ in (10, 25) if m_ < n]
    for k in ks:
        for m_ in ms:
            hit = sum(pos[x] <= m_ for x in top[:k])
            print(f"  ベースライン上位{k:<3} が モデル上位{m_:<3} に: "
                  f"{hit}/{min(k, len(top))}   偶然 {m_ / n:.0%}")
    if not (ks and ms):
        print("  候補数が少なすぎるので上位k比較は省略します。")
elif rows:
    print("BASELINE_JSON が無いので照合は省略します。")


### 文献頻度との相関（頻度バイアスの確認）

段階 1（PMI）が動いていないバックエンドでは、出現頻度を補正するものが何もありません。
遺伝子ごとの PubMed 掲載数の JSON があれば、ここで確認できます。


In [ ]:
FREQ_JSON = None    # 例: {"TP53": 20400, ...} の JSON へのパス

if rows and FREQ_JSON and os.path.exists(FREQ_JSON):
    import math
    freq = json.load(open(FREQ_JSON))
    g = [r["gene"] for r in rows]
    m = [r["mean_score"] for r in rows]
    f = [math.log1p(freq.get(x, 0)) for x in g]
    print(f"Spearman(平均スコア, log 文献数) = {spearman(m, f):+.3f}")
    print("0 から離れているほど、疾患ではなく知名度を読んでいます。")
else:
    print("FREQ_JSON 未設定。段階 1 が使えないバックエンドでは、ここを埋めて確認してください。")


## 8. 保存


In [ ]:
OUT_JSON = f"{DISEASE.replace(' ', '_')}_repeats.json"
OUT_CSV  = f"{DISEASE.replace(' ', '_')}_repeats.csv"

if rows:
    with open(OUT_JSON, "w") as fh:
        json.dump({
            "disease": DISEASE, "model": MODEL, "backend": BACKEND,
            "repeats": REPEATS, "seed": SEED, "n_genes": len(GENES),
            "deterministic_probe": det,
            "instability": instability(rows),
            "runs": per_run, "rows": rows,
        }, fh, indent=1)

    cols = ["rank", "gene", "mean_score", "sd_score",
            "mean_rank", "best_rank", "worst_rank", "rank_sd", "n_runs"]
    with open(OUT_CSV, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    print("書き出しました:", OUT_JSON, "/", OUT_CSV)
else:
    print("結果がないので保存しません。")


## 9. 数字を信じる前に

- **平均化は不安定さを隠しません。減らすだけです。** SD と順位変動幅を必ず一緒に読んでください。
  順位変動幅の中央値が数十位ある状態では、「上位 25 の集合」までしか主張できません。
- **繰り返しはグループ構成のばらつきしか均しません。** 疾患の読み違い、系統的な文献頻度バイアス、
  同族遺伝子の取り違えは、何回まわしても消えません。7 章の照合と `evaluate.py --controls` が要ります。
- **棄権を尊重してください。** 判定を返さなかった回が多いなら、候補セットに対して
  モデルが 1 位を決めきれていないということです。上位を無理に読むより、候補を絞るか
  段階 1（PMI）が使えるバックエンドに替えるほうが確実です。
- 段階 1 を有効にすれば、そもそも候補が 10〜20 件に絞られてからグループ分けされるので、
  この不安定さ自体が小さくなります（`--backend llamacpp`）。
